# 05 · Live vs backtest — where the LONG edge goes, and what could make LONG profitable

**Question.** The backtest said LONG makes ~+0.7% of notional per trade. Three months of live paper
trading lost money. Is it the entry, the timing, the stock, or the exits — and is there a *conservative*
LONG configuration that is actually profitable?

**Scope.** LONG only (shorts are paused). All money figures are at **₹2,00,000 per trade** (the planned live
capital), after brokerage, STT, exchange, GST, stamp duty and 0.05%/side slippage.

**Data.** Live paper trades 2026-06-15 → 09-18 (66 LONG), every live entry decision from the session logs
(06-29 → 09-18), 5-min bars for ~500 stocks, and a **causal replay** of the live entry logic on 426 trading
days (2025-01 → 2026-09) — what live would have done each day, using only bars that existed at the time.

**How to run.** Kernel *TRAIAGENT (venv)*. The replay is precomputed in `reports/gap_analysis/replay/`
(re-create with `python -m analysis.run_replay`, ~hours). This notebook then runs in a few minutes.

## 0 · Answers


**Short version: the backtest's LONG profit was mostly hindsight. The rules live can actually run lose about
0.24% per trade — in 2025 as well as 2026 — and none of the 26 conservative variants tested makes LONG profitable.
Do not put ₹2L into this LONG system yet.**

| Your question | Answer (425 trading days, 2025-01 → 2026-09, ₹2L per trade, after costs) |
|---|---|
| Is **the stock** the problem? | **Yes — the biggest part.** The backtest chooses the stock using votes cast *later in the day*: its picks have ~13 agreeing strategies by the close but only ~7 at their own entry, and **41% would not have passed the quality filters when it "entered"**. Those late votes (STOCHASTIC, ORB, REL-STR = "outperformed NIFTY today"…) fire *because the stock went up*, so their "predictive power" is the day's outcome itself. |
| Is **the entry** the problem? | **Yes.** The backtest buys at the strategy's level, but the signal only confirms after the price has moved past it — typically +0.26% (a quarter of the time more than +0.5%). |
| | Together these two explain almost all of the **0.92%/trade gap** (backtest +0.68% → causal −0.24%). |
| Is **timing** the problem? | No. Skipping the first 20–35 minutes, waiting for more agreement, the ~3-minute scan latency — none of it changes the sign. |
| Is **the target / exit** the problem? | No. The backtest's target-before-stop is worth ~0.02%; every exit tested, including your **+1% / −1%**, lands between −0.20% and −0.26%. |
| **First-pass stock + limit order?** | It loses less per trade (−0.16% to −0.19%) only because it trades on half the days: the limit fills when the breakout **fails** (those fills win 28%, 68% get stopped) and misses the real breakouts (which would have won 59%). |
| Why did live do even worse recently? | Live was **not running the tested system**: the volume feed was dead all of July (92% of bars had zero volume) and ~20% broken after, and the 09:15 bar's open was wrong. On the stable-config days live lost −0.45%/trade; the same rules on correct data lost ~−0.07% — a small sample (±0.3%), so the feed *may have added* losses, on top of the main problem. |
| Is there a **conservative, profitable** LONG setup? | **Not in this signal set.** The best of 26 loses −0.13%/trade, which is about +0.03% *before* costs — i.e. no skill. |

**What to do** (§11): keep LONG paper trading paused and don't fund it; fix the research tool (use this causal replay
as the backtester) and live data parity; then research a **multi-day swing LONG** (decide on the previous close, enter
at the open, hold 3–10 days), where 0.16% of costs is small against the move.

## 1 · Setup

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)

from analysis import report as rep
from analysis.live_log_parser import load_decisions, first_pass_candidates
from analysis.policies import (load_replay, trades_from_rows, bt_trades, pick_live, pick_first_pass, summarize,
                               ExitRule, LIVE_EXIT, BT_EXIT, REAL_BT_EXIT)
from analysis.gap import waterfall
from analysis import scenarios as sc
from analysis.validate import validate

NOTIONAL = 200_000
L   = rep.live_long()                                 # live LONG paper trades
dec = load_decisions(ROOT / "logs" / "ec2")           # every logged live entry decision
bt, cand = load_replay()                              # backtest picks + causal live-replay candidates
print(f"live LONG trades: {len(L)} ({L.date.min():%Y-%m-%d} -> {L.date.max():%Y-%m-%d})")
print(f"logged decision events: {len(dec):,} over {dec.date.nunique()} sessions")
print(f"replay: {cand.date.nunique()} days ({cand.date.min():%Y-%m-%d} -> {cand.date.max():%Y-%m-%d}), "
      f"{len(cand):,} candidate rows, {len(bt)} backtest picks")

live LONG trades: 66 (2026-06-15 -> 2026-09-18)
logged decision events: 2,060 over 65 sessions
replay: 425 days (2025-01-01 -> 2026-09-18), 446,250 candidate rows, 426 backtest picks


## 2 · How the LONG decision is actually made


Both the backtest and live use the same machinery; the difference is **when** they are allowed to look.

1. **Watchlist** (before 09:15, history only): ~200 stocks by trend / chart patterns / liquidity.
2. **Signals**: 37 strategies each say LONG / SHORT / nothing, with an entry *level*, stop and target.
3. **Score** = sum of the *weights* of the strategies saying LONG. Live and backtest both load
   `checkpoints/wf5_weights.json`. In practice this is a **vote tally**: 21 of 37 LONG weights sit at the
   3.0 cap and 10 at 1.0, so the score is ~3 points per strong strategy. (The regime modifiers never change
   anything — VIX is hard-coded to 15 — and the NIFTY-gap bias multiplies every LONG score equally, so it never
   reorders LONG candidates.) So yes: it is essentially **signal firing + counting**, with the weights only
   separating "strong" from "weak" voters.
4. **Driver** = the voting strategy with the best *lifetime win-rate × RR*; its level/stop/target define the trade.
   Two strategies vote without an entry (ADX-FILTER, DAILY-BIAS) — they add score and agreement but never drive.
5. **Quality filters**: ≥4 agreeing strategies (lifetime LONG win-rate ≥40%), RR ≥1.5, liquidity ≥₹50 Cr/day,
   time gates (signals from 09:30 need 5 agreeing; from 10:30 a higher score), skip the 50–55% "danger zone".

| | Backtest (`backtester/engine.py`) | Live (`live/live_engine.py`) |
|---|---|---|
| bars seen | the **whole day** | only bars closed so far (every 5 min) |
| stock ranking, agreement, time gate | on the **whole day's** votes | on votes cast so far |
| entry | **at the strategy level**, at signal_time | at the live price ~3 min after the bar (scan latency) |
| extra gates | none | 30-min signal expiry, 0.30% drift gate, RR re-check, stop viability, fill gate |
| exits | target checked before stop, 15:15 | ticks, 1% stop cap, trailing lock, 14:50 |

In [2]:
import json
w = json.load(open(ROOT / "checkpoints" / "wf5_weights.json"))
wl = pd.Series({k: v["long"] for k, v in w.items()}).round(3)
print("LONG weight values (count of strategies):", wl.value_counts().sort_index(ascending=False).to_dict())
from weights.regime import get_regime_modifiers
print("regime modifiers at VIX=15 (VIX is hard-coded to 15):",
      {k: v for k, v in get_regime_modifiers(w, vix=15.0).items() if v != 1.0} or "all 1.0 — inert")

LONG weight values (count of strategies): {3.0: 21, 2.531: 1, 1.0: 10, 0.633: 1, 0.534: 1, 0.5: 1, 0.25: 1, 0.125: 1}
regime modifiers at VIX=15 (VIX is hard-coded to 15): all 1.0 — inert


## 3 · The three-month live LONG record


66 LONG trades, −₹37,921 as sized live (~₹5L notional). Two things to read off the tables:

* **The first three weeks flatter the total.** Until 07-13 an early build chased entries at the live price and exited
  at 15:15; those 19 trades made +₹21k. Everything since has lost money, and the **current configuration (every gate,
  14:50 exit, from 07-31) is the worst: about −0.45% of notional per trade** (34 trades).
* **Stops are the whole loss**: STOP_HIT trades lose ~₹1.37L; targets and profit-locks recover only part of it.

In [3]:
print("By month"); display(rep.by(L, "month"))
print("By exit reason"); display(rep.by(L, "exit_reason"))
print("By live configuration in force (the sample mixes 8 configurations)"); display(rep.by(L, "era"))
fig = go.Figure(go.Scatter(x=L.date, y=L.sort_values("date").pnl_rs.cumsum(), mode="lines+markers"))
fig.update_layout(title="Live LONG cumulative P&L (paper, as sized live)", height=320, template="plotly_white")
fig.show()

By month


,trades,win_pct,net_rs,avg_ret_pct
month,,,,
2026-06,10,40.0,22406,0.444
2026-07,23,39.1,4319,0.036
2026-08,20,45.0,-40432,-0.442
2026-09,13,23.1,-24214,-0.425


By exit reason


,trades,win_pct,net_rs,avg_ret_pct
exit_reason,,,,
PROFIT_LOCK_STOP,13,92.3,24070,0.423
STOP_HIT,30,0.0,-136598,-0.976
TARGET_HIT,9,88.9,79713,1.779
TIME_EXIT,14,35.7,-5105,-0.095


By live configuration in force (the sample mixes 8 configurations)


,trades,win_pct,net_rs,avg_ret_pct
era,,,,
"2026-06-15 early build: re-anchored entries to live price (chased), 15:15 exit",19,31.6,21004,0.215
2026-07-14 drift gate 0.30% + 30-min expiry + trailing profit-lock,4,50.0,-1637,-0.082
2026-07-20 ATR-normalised sizing,3,66.7,7154,0.511
2026-07-23 stop-viability gate,4,50.0,-2843,-0.159
2026-07-29 ride past target once locked,1,100.0,7524,1.518
2026-07-30 final stop cap 1.5%,1,0.0,-564,-0.113
2026-07-31 fill gate (cross the spread) + square-off 14:50,25,36.0,-57589,-0.512
2026-09-06 stop cap tightened to 1.0%,9,33.3,-10970,-0.261


## 4 · Live was not running the backtested system — data problems


Before comparing results, note that for much of the sample **live was not fed the same data the backtest used**:

* **Volume feed.** In **July 92% of live-built 5-min bars had zero volume** (100% on most days); from 08-10 about
  20% still did, with the volume lumped into catch-up bars (the first-hour total matches history, ~0.99×), which
  fakes volume spikes. Every volume-gated strategy (VPOC, PDH-PDL, ORB, SR-BREAK, VWAP…) voted differently live.
* **Opening bar.** The live 09:15 bar starts from the first streamed tick, not the exchange open: median
  0.20–0.25% off. Every FIRST-CANDLE level is computed from that bar.
* **Parquet time zones.** 2024–25 bars are tz-aware (+05:30), 2026 bars are tz-naive. `run_testing.py 2026`
  today would **silently drop every 2026 file** (the concat fails inside a `try/except: pass`). The June backtests
  predate the change, so historical results are unaffected — but nobody can re-run 2026 until this is fixed.
* **Duplicated backtest rows.** `paper_trades_full_history.csv` has 2025 appended twice (442 duplicate rows);
  the July divergence notebook (04) double-counted 2025. The clean WF-6 run is `paper_trades.csv`.
* **Trade-log caveat.** For PROFIT_LOCK_STOP trades, `stop_loss` in `live_paper_trades.csv` is the *final
  trailed* stop, not the original.
* **Eight configurations in three months.** The live sample mixes an early build that chased entries (to 07-13)
  with seven later rule changes — the table in section 3 splits results by the configuration in force.

The replay below uses the **historical** bars (correct volume, correct open), i.e. it answers *"what would live
have done with correct data"*.

In [4]:
H = rep.live_candle_health(); H["month"] = H.date.str[:7]
print("Live-built 5-min bars with ZERO volume (to 11:00), by month:")
display(H.groupby("month").agg(sessions=("date", "size"), zero_volume_pct=("zero_volume_pct", "mean")).round(1))

import pandas as pd
for y in (2025, 2026):
    d = pd.read_parquet(ROOT / "data" / "stocks" / str(y) / "TECHM.parquet", columns=["datetime"])
    print(f"{y} parquet datetime dtype: {d.datetime.dtype}")
print("-> backtester.engine._preload_data concatenates the two inside try/except: the 2026 file is silently dropped.")

full = pd.read_csv(ROOT / "data" / "trade_logs" / "ec2_snapshot" / "paper_trades_full_history.csv", low_memory=False)
print(f"paper_trades_full_history.csv: {len(full)} rows, {full.duplicated(['date','symbol','direction']).sum()} duplicated (date, symbol, direction)")

Live-built 5-min bars with ZERO volume (to 11:00), by month:


,sessions,zero_volume_pct
month,,
2026-06,2,81.8
2026-07,25,92.0
2026-08,23,17.8
2026-09,15,21.3


2025 parquet datetime dtype: datetime64[ns, pytz.FixedOffset(330)]
2026 parquet datetime dtype: datetime64[us]
-> backtester.engine._preload_data concatenates the two inside try/except: the 2026 file is silently dropped.
paper_trades_full_history.csv: 2905 rows, 442 duplicated (date, symbol, direction)


## 5 · Where the backtest sees the future


Three separate kinds of hindsight exist in `backtester/engine.py`. Section 5.1 tests the signals themselves;
5.2 the stock selection; 5.3 the entry price.


### 5.1 Are the strategies' signals causal?

Each strategy was run on the full day and on the day truncated after every bar (what live sees), on 180 random
stock-days (`python -m analysis.causality_audit`). **Most signals are honest**: CAMARILLA, FIRST-CANDLE, VPOC,
RSI-EXT, BOLLINGER, MACD, SUPERTREND, PDH-PDL, CPR… are 100% causal. A minority are not: PIN-BAR, SR-BREAK,
EMA-CROSS (back-dated by a median 80 min), VWAP-REV and ORB-15/30 stamp signals earlier than they could be
known; INTRADAY-STRUCT stamps the last bar; and **ADX-FILTER votes on the *last* bar's ADX** — on a full day that
is the 15:25 ADX, i.e. "did this stock end the day in a strong uptrend". It is weight 3.0 and counts toward
agreement. (The reviewer found it gains and loses about as many votes as it adds, so its net effect is small.)
The replay recomputes all of these bar by bar.

In [5]:
display(rep.causality_table())

cls,fired,exact,delayed,median_backdate_min,unstable,never,late_or_untimed,causal_%
strategy,,,,,,,,
OPEN-WEAK,24,0,24,5.0,0,0,0,0.0
INTRADAY-STRUCT,56,0,0,0.0,0,0,56,0.0
PIN-BAR,135,20,87,35.0,18,10,0,15.0
SR-BREAK,110,25,60,15.0,21,4,0,23.0
FAILED-BO,53,13,28,22.5,12,0,0,25.0
BEAR-ENGULF,32,14,16,57.5,1,1,0,44.0
EMA-CROSS,23,14,7,80.0,2,0,0,61.0
VWAP-REV,49,42,7,10.0,0,0,0,86.0
ORB-30,80,69,11,5.0,0,0,0,86.0


### 5.2 Did the backtest's pick even qualify at its own signal time?

For each of 401 backtest picks, the replay shows the stock at the first scan that could have seen its signal, counting
only votes cast by then. **Median agreement: 13 strategies by the close, 7 at entry time; median score 39 vs 19. 41% of
the picks would not have passed the quality filters at that moment** (mostly the 50–55% danger zone, not yet enough
agreement, or not even a scored candidate). The backtest took them because of votes that arrived later. Those late
voters (STOCHASTIC, ORB-15/30, REL-STR — literally "outperformed NIFTY today" — SR-BREAK) fire *because the stock went
up*, so the backtest was in effect picking the day's winners after the fact. The reviewer found these picks rise
~+0.7% from the next bar's open to the close in 16 of 18 months — even in a month when NIFTY fell 11% — while a random
liquid stock over the same window returns about −0.05%.

In [6]:
X = rep.bt_pick_at_signal_time(bt, cand)
X = X[X.scan <= "11:00"]
print(f"backtest picks with signal <= 10:55: {len(X)}")
print(f"  full-day agreeing strategies: median {X.full_day_agreeing.median():.0f} | at signal time: median {X.agreeing_then.median():.0f}")
print(f"  full-day score:               median {X.full_day_score.median():.1f} | at signal time: median {X.score_then.median():.1f}")
print(f"  would NOT have passed the quality filters at their own signal time: {100*(~X.quality_then).mean():.0f}%")
print("  why they fail at signal time:"); display(X[~X.quality_then].reason_then.str.replace(r"[\d.]+", "#", regex=True).value_counts().head(6))

backtest picks with signal <= 10:55: 401


  full-day agreeing strategies: median 13 | at signal time: median 7
  full-day score:               median 39.0 | at signal time: median 18.6
  would NOT have passed the quality filters at their own signal time: 41%
  why they fail at signal time:


reason_then
Predicted win pct #% in #-#% danger zone                                         94
not a scored candidate at that scan                                              31
Only # strategy agreed (need #)                                                  30
Signal at #:# requires score >= # and #+ agreements (got score=#, agreeing=#)     9
Name: count, dtype: int64

### 5.3 Entry at the level vs the price you can actually get

The backtest buys at the strategy *level* (the first candle's high, Camarilla H4, the VPOC…), but the signal only
confirms when a bar **closes** past that level, and live then needs ~3 minutes to scan. At the first scan that could act,
the price is **median +0.26% above the level (mean +0.36%; a quarter of the time over +0.5%, a tenth over +0.8%)**.
The backtest books every one of those trades at the level.

In [7]:
T_lvl = bt_trades(bt, "bt_level", REAL_BT_EXIT, NOTIONAL)
T_mkt = bt_trades(bt, "market", REAL_BT_EXIT, NOTIONAL)
print(f"price at the first scan that could see the signal vs the strategy level (backtest picks):")
print(T_mkt.drift_pct.describe(percentiles=[.25, .5, .75, .9]).round(3).to_string())

price at the first scan that could see the signal vs the strategy level (backtest picks):
count    410.000
mean       0.364
std        0.496
min       -0.534
25%        0.091
50%        0.256
75%        0.508
90%        0.798
max        6.049


## 6 · Is the replay faithful to live?


Three checks against the live session logs, split by the state of live's own data feed. The replay reproduces live's
**gate logic exactly** (re-run on the replay's own price it gives identical verdicts on all 63,000 rows checked and the
same daily pick every day); the simulator reproduces **94% of live exit reasons** (per-trade P&L correlation 0.92;
≈0.1%/trade optimistic, because live stops fill a tick below the level); and the backtest step reproduces the engine's
own exits on 100% of its picks. Where replay and live differ is the **inputs**: agreement with live's logged scores rises
from ~30% in July (live volume feed dead) to ~56% once the feed was restored — the replay runs on correct bars, live
did not. Daily picks also sit on knife edges (a 0.2% price move flips the viability gate; a 0.2-point win-% flips the
danger zone), so live and replay are compared by distribution, not day by day.

The last line: on the stable-config days since 07-31, live lost −0.45%/trade (95% CI −0.68..−0.21) while the same rules
on correct data lost ~−0.07% (35 trades; before the ~0.1% simulator optimism). With ~30 trades each this is only
~1–1.5 standard errors, so the broken feed **may have added** losses on top of the edge problem — it is not the main story.

In [8]:
per = {"Jul (live volume feed dead)": ("2026-06-29", "2026-07-31"), "Aug 03-07 (volume restored)": ("2026-08-01", "2026-08-09"),
       "Aug 10+ (~20% zero-volume bars)": ("2026-08-10", "2026-09-18")}
rows = []
for name, (a, b) in per.items():
    c = cand[(cand.date >= a) & (cand.date <= b)]
    v = validate(c, dec[(dec.date >= a) & (dec.date <= b)])
    rows.append({"period": name, "days": c.date.nunique(),
                 "score exact %": v["inputs"]["score_exact"], "agreement exact %": v["inputs"]["agreeing_exact"],
                 "PASS/FAIL same %": v["inputs"]["pass_fail_same"], "same gate %": v["gates"]["same_gate_pct"],
                 "level exact %": v["gates"]["level_exact_pct"], "same stock %": v["picks"]["same_stock_pct"]})
display(pd.DataFrame(rows).set_index("period"))

,days,score exact %,agreement exact %,PASS/FAIL same %,same gate %,level exact %,same stock %
period,,,,,,,
Jul (live volume feed dead),25,29.1,30.9,70.9,34.9,66.7,3.8
Aug 03-07 (volume restored),5,56.2,58.3,89.6,56.8,81.1,20.0
Aug 10+ (~20% zero-volume bars),29,44.1,45.3,80.7,54.6,85.5,28.6


In [9]:
# Distribution check: replay 'live rules' vs actual live, same days, same exits, Rs 2L
live_days = L[L.date >= "2026-07-31"]
rep_live = trades_from_rows(pick_live(cand[(cand.date >= "2026-07-31")]), "market", LIVE_EXIT, NOTIONAL)
act = live_days.assign(net_pct=live_days.pnl_rs / live_days.notional * 100)
se = act.net_pct.std() / np.sqrt(len(act))
print(f"actual live (stable config, from 07-31): n={len(act)}  mean {act.net_pct.mean():+.3f}%/trade (95% CI {act.net_pct.mean()-1.96*se:+.2f}..{act.net_pct.mean()+1.96*se:+.2f})  win {100*(act.pnl_rs>0).mean():.0f}%")
s_ = summarize(rep_live, "replay")
print(f"replay of live's rules on CORRECT data:  n={s_['trades']}  mean {s_['avg_net_pct']:+.3f}%/trade (t {s_['t_stat']:+.2f})  win {s_['win_pct']:.0f}%")

actual live (stable config, from 07-31): n=34  mean -0.446%/trade (95% CI -0.68..-0.21)  win 35%
replay of live's rules on CORRECT data:  n=35  mean -0.068%/trade (t -0.50)  win 54%


## 7 · The gap, step by step


Each step changes exactly one thing, on the same days, at ₹2L per trade after costs:

* **W0 → W1 honest exits** (stop-first on bars that touch both, 14:50): about −0.02%. *Exits are not the problem.*
* **W1 → W2 honest entry price** (buy after the signal confirms, not at the level): **−0.36%**.
* **W2 → W3 no hindsight in picking** (rank and filter only on votes cast so far): **−0.51%**.
* **W3 → W5 live's own gates and exits**: about −0.03% overall (helps in some periods, hurts in others).

The split between the two big steps depends on their order (W2 is measured on the hindsight picks); the robust
statement is that **together they explain almost the whole 0.92%/trade gap**, in every period. **W5 is what live can
actually do: −0.24% per trade (t −5.5) — −0.28% in 2025, −0.15% in 2026 H1, −0.26% in the live window.** It is not the
2026 market, and it matches what live actually lost (−0.25% per trade from 06-29).

In [10]:
W, steps = waterfall(bt, cand, NOTIONAL)
per = {**sc.PERIODS, "ALL 2025-01..2026-09": (None, None)}
rows = []
for k, t in steps.items():
    for p, (a, b) in per.items():
        tt = t if a is None else t[(t.date >= a) & (t.date <= b)]
        rows.append({"step": k, "period": p, **{kk: vv for kk, vv in summarize(tt, k).items() if kk != "policy"}})
WF = pd.DataFrame(rows)
display(WF.pivot_table(index="step", columns="period", values="avg_net_pct", sort=False))
display(WF[WF.period == "ALL 2025-01..2026-09"].set_index("step").drop(columns="period"))

period,2025 (choose),2026 H1 (check),live window,ALL 2025-01..2026-09
step,,,,
W0 backtest as reported,0.616,0.749,0.824,0.684
W1 + honest exits,0.597,0.697,0.800,0.655
W2 + honest entry price,0.251,0.302,0.469,0.297
W3 + no hindsight in picking,-0.232,-0.315,-0.001,-0.215
W4 + live's own gates,-0.281,-0.183,-0.121,-0.231
W5 + live's exit policy,-0.278,-0.155,-0.260,-0.244


,trades,win_pct,net_rs,avg_rs,avg_net_pct,t_stat,max_dd_rs,stop_pct
step,,,,,,,,
W0 backtest as reported,426,64.1,576795,1354,0.684,10.74,-21070,24.2
W1 + honest exits,426,66.0,553801,1300,0.655,10.61,-22250,23.5
W2 + honest entry price,410,58.5,241309,589,0.297,5.41,-29554,23.9
W3 + no hindsight in picking,407,39.3,-174714,-429,-0.215,-3.62,-182242,44.5
W4 + live's own gates,425,35.3,-196533,-462,-0.231,-3.99,-208583,44.5
W5 + live's exit policy,425,40.0,-206005,-485,-0.244,-5.54,-206504,47.5


## 8 · The entry decision: which stock, when, and at what price

### 8.1 What live skipped, and what the skipped stocks did

On correct data, the first stock live "wanted" each day (passed expiry, geometry and quality, price still between stop
and target) was **drift-skipped on 296 of 425 days (70%)**, viability-skipped on 82 and taken on only 31. Live then
walks down the list to a lower-ranked stock whose price happens to still be near its level.

In [11]:
feat = sc.add_features(cand, cache=ROOT / "reports" / "gap_analysis" / "features.parquet")   # tradeable candidates + causal features
first = pick_first_pass(cand)
gate_of_first = first.gate_c.value_counts()
print("first quality-passing candidate each day — what live's gates did with it:"); display(gate_of_first)

first quality-passing candidate each day — what live's gates did with it:


gate_c
SKIP_DRIFT        296
SKIP_VIABILITY     82
TAKE               31
SKIP_RR            16
Name: count, dtype: int64

### 8.2 The drift gate — does skipping 'chased' prices help?

Measured **causally** (price at the decision vs the level; using the bar-mid here would peek at the bar's close — a bug
the reviewer caught in my first version), the bucket live takes (0–0.3%) returns −0.22%; moderate drift (0.3–1.0%)
−0.12% to −0.14%; over 1% −0.45%. No bucket is profitable: the gate does not select better trades, it just changes
which losing trade you take.

In [12]:
fp = feat.sort_values(["date", "scan", "rank"]).groupby("date").head(1)
T = trades_from_rows(fp, "market", LIVE_EXIT, NOTIONAL)
T["net_pct"] = T.pnl / (T.entry * T.qty) * 100
T["drift_bkt"] = pd.cut(T.drift_pct, [-5, 0, 0.3, 0.6, 1.0, 5], labels=["<=0 (at/below level)", "0-0.3 (live takes)", "0.3-0.6", "0.6-1.0", ">1.0"])
print("drift measured CAUSALLY (scan bar's open vs the strategy level); entry at bar-mid; live exits")
display(T.groupby("drift_bkt", observed=True).agg(trades=("pnl", "size"), win_pct=("pnl", lambda x: round(100*(x>0).mean(),1)),
        avg_net_pct=("net_pct", "mean"), net_rs=("pnl", "sum")).round(3))

drift measured CAUSALLY (scan bar's open vs the strategy level); entry at bar-mid; live exits


,trades,win_pct,avg_net_pct,net_rs
drift_bkt,,,,
<=0 (at/below level),16,43.8,0.090,2514.624
0-0.3 (live takes),112,33.9,-0.222,-48677.631
0.3-0.6,103,46.6,-0.118,-24555.962
0.6-1.0,70,52.9,-0.136,-19160.850
>1.0,106,34.0,-0.445,-93759.390


### 8.3 Your idea: take the first-pass stock with a LIMIT order at the strategy level

Your idea: instead of skipping the first-pass stock when the price has run, rest a **buy limit at the strategy level**.
The numbers show the trap: the limit fills on **46% of days**, and those fills **win only 28% (68% stopped out)** — it
fills precisely when the breakout fails and the price comes back. The days it misses are the real breakouts: bought at
market, **those would have won 59%** (+0.07% per trade). The loss per trade shrinks only because you trade less often.

In [13]:
lim = trades_from_rows(fp, "limit", LIVE_EXIT, NOTIONAL, limit_window_bars=6)
mkt = trades_from_rows(fp, "market", LIVE_EXIT, NOTIONAL)
print(f"first-pass candidates: {len(fp)} | limit filled within 30 min: {len(lim)} ({100*len(lim)/len(fp):.0f}%)")
display(pd.DataFrame([summarize(mkt, "first-pass, market"), summarize(lim, "first-pass, limit at level (30 min)")]))
filled = set(zip(lim.date, lim.symbol)); unf = mkt[[ (d, s) not in filled for d, s in zip(mkt.date, mkt.symbol)]]
display(pd.DataFrame([summarize(unf, "…the ones the limit MISSED, had you bought at market")]))

first-pass candidates: 425 | limit filled within 30 min: 195 (46%)


,policy,trades,win_pct,net_rs,avg_rs,avg_net_pct,t_stat,max_dd_rs,stop_pct
0,"first-pass, market",407,40.8,-183639,-451,-0.227,-5.41,-183743,51.6
1,"first-pass, limit at level (30 min)",195,27.7,-72013,-369,-0.186,-3.69,-75877,68.2


,policy,trades,win_pct,net_rs,avg_rs,avg_net_pct,t_stat,max_dd_rs,stop_pct
0,"…the ones the limit MISSED, had you bought at ...",213,59.2,29684,139,0.071,1.22,-10035,29.1


### 8.4 Timing and stock features

Known-at-decision features, on the first-pass stock each day. **NIFTY vs its open**: every bucket negative (best,
NIFTY above +0.2%, still −0.13%). **Run-up from the stock's own open**: negative in every bucket except over 2% (25
trades — too few to use). **Driver**: CAMARILLA −0.22%, VPOC −0.25%. (With correct volume VPOC drives almost half the
picks; live's dead volume feed silenced it — one reason live's picks differ from the replay's.) Timing is tested as
rules in §9.

In [14]:
T = trades_from_rows(fp, "market", LIVE_EXIT, NOTIONAL).merge(fp[["date", "symbol", "runup_pct", "nifty_now_pct", "vwap_pct"]], on=["date", "symbol"])
T["net_pct"] = T.pnl / (T.entry * T.qty) * 100
def tab(col, bins, labels):
    T["b"] = pd.cut(T[col], bins, labels=labels)
    return T.groupby("b", observed=True).agg(trades=("pnl", "size"), win_pct=("pnl", lambda x: round(100*(x>0).mean(),1)), avg_net_pct=("net_pct", "mean")).round(3)
print("by NIFTY vs its open at the scan"); display(tab("nifty_now_pct", [-9, -0.2, 0, 0.2, 9], ["< -0.2%", "-0.2..0", "0..+0.2", "> +0.2%"]))
print("by stock run-up from its open");     display(tab("runup_pct", [-9, 0, 0.5, 1, 2, 9], ["< 0", "0-0.5", "0.5-1", "1-2", "> 2"]))
print("by driver"); display(T.groupby("driver").agg(trades=("pnl", "size"), avg_net_pct=("net_pct", "mean")).round(3).sort_values("trades", ascending=False).head(8))

by NIFTY vs its open at the scan


,trades,win_pct,avg_net_pct
b,,,
< -0.2%,81,44.4,-0.198
-0.2..0,127,37.0,-0.295
0..+0.2,126,38.9,-0.230
> +0.2%,73,46.6,-0.134


by stock run-up from its open


,trades,win_pct,avg_net_pct
b,,,
< 0,94,31.9,-0.315
0-0.5,113,39.8,-0.215
0.5-1,97,44.3,-0.202
1-2,78,43.6,-0.256
> 2,25,56.0,0.048


by driver


,trades,avg_net_pct
driver,,
CAMARILLA,199,-0.217
VPOC,184,-0.254
GAP-CONT,11,-0.120
FIRST-CANDLE,8,-0.115
CPR,3,-0.474
GAP-FADE,2,0.624


## 9 · Scenarios — is there a conservative LONG configuration that makes money?


Every scenario uses only information available at the decision, is priced at ₹2L per trade after costs, and is
reported separately for **2025** (used to *choose*), **2026 H1** (a check the rule was not chosen on) and the **live
window** (Jun 15 – Sep 18, the period your paper trades cover). A rule is only interesting if it is **positive in all
three** — with one trade a day, a per-trade spread of ~1% means a mean of +0.1% needs ~400 trades to reach t≈2.
The simulator is ~0.1%/trade optimistic vs live, so treat anything below ~+0.1% as zero.

Groups: **A** what live does vs taking the first-pass stock (market or your limit order) · **B** the drift gate,
both ways · **C** a NIFTY market filter · **D** dropping FIRST-CANDLE as driver · **E** exits, including your
+1% / −1% plan · **T** timing (skip the opening minutes) · **F** wait until more strategies agree.

In [15]:
from analysis.scenario_specs import SPECS
R, TR = sc.run(feat, cand, SPECS, NOTIONAL)
R.to_csv(ROOT / "reports" / "gap_analysis" / "scenario_results.csv", index=False)
def cell(r):
    return "—" if not r.trades else f"{r.avg_net_pct:+.2f}% (t {r.t_stat:+.1f}, n {r.trades})"
G = R.assign(v=R.apply(cell, axis=1)).pivot_table(index="policy", columns="period", values="v", aggfunc="first", sort=False)
G = G[list(sc.PERIODS)]
G["all three > 0"] = [all(R[(R.policy == p) & (R.period == q)].avg_net_pct.fillna(-1).iloc[0] > 0 for q in sc.PERIODS) for p in G.index]
print("mean net return per trade after costs (t-stat, trades) — Rs 2L notional; sim is ~0.1%/trade optimistic vs live")
display(G)

mean net return per trade after costs (t-stat, trades) — Rs 2L notional; sim is ~0.1%/trade optimistic vs live


period,2025 (choose),2026 H1 (check),live window,all three > 0
policy,,,,
"A1 live rules (all gates), live exits","-0.28% (t -5.1, n 248)","-0.15% (t -1.5, n 109)","-0.26% (t -2.7, n 68)",False
"A2 first-pass stock, market, live exits","-0.20% (t -3.8, n 241)","-0.29% (t -3.3, n 100)","-0.21% (t -2.1, n 66)",False
"A3 first-pass stock, LIMIT at level 30m, live exits","-0.24% (t -4.3, n 113)","-0.14% (t -1.2, n 56)","-0.04% (t -0.2, n 26)",False
"A4 first-pass stock, LIMIT at level 60m, live exits","-0.22% (t -4.0, n 129)","-0.13% (t -1.2, n 61)","+0.04% (t +0.2, n 30)",False
"B1 first-pass, drift < 0.3% (live's gate only)","-0.14% (t -2.9, n 242)","-0.11% (t -1.3, n 107)","-0.14% (t -1.5, n 64)",False
"B2 first-pass, drift >= 0.3% (inverted gate)","-0.22% (t -3.9, n 240)","-0.31% (t -3.7, n 101)","-0.18% (t -1.7, n 66)",False
"B3 first-pass, drift >= 0.5%","-0.18% (t -2.9, n 238)","-0.36% (t -4.3, n 101)","-0.24% (t -2.2, n 66)",False
"C1 first-pass, NIFTY >= +0.2% vs open","-0.17% (t -2.1, n 111)","-0.37% (t -3.5, n 54)","-0.16% (t -0.9, n 22)",False
"C2 first-pass, NIFTY >= 0 vs open","-0.19% (t -3.1, n 177)","-0.23% (t -2.5, n 76)","-0.06% (t -0.4, n 42)",False


**All 26 lose money overall; none is positive in all three periods; most are significantly negative.**

* Your **fixed +1% / −1%**: −0.23% (first-pass stock) / −0.26% (live's picks).
* Your **first-pass stock + limit at the level**: −0.19% (30 min) / −0.16% (60 min), on only about half the days.
* The reviewer's **inverted drift gate** (withdrawn after its own audit): −0.24%. **NIFTY filters**: −0.17% to −0.26%.
* **Timing** (skip the first 20 / 35 minutes): −0.19% / −0.15%. **Wait for ≥9 agreeing + NIFTY ≥ 0**: −0.13% (best).

Why no tweak works — the cost arithmetic: a round trip at ₹2L costs **0.16%** (0.10% of it assumed slippage). The best
variant, −0.13% net, is **≈ +0.03% before costs**; buying *any* liquid stock at 09:25 returns about −0.20% net (the
reviewer's universe check). The filters beat random by ~0.07%. **Once hindsight is removed, this signal set has
essentially no intraday stock-picking skill** — this is not a parameter problem.

## 10 · Independent review (subagent) — where we agree and disagree


An independent reviewer (a separate agent, read-only on the code) went through three rounds.

**Round 1 — its own analysis of the gap.** It put the gap at 0.9–1.1%/trade (I had said 0.7%: the first June trades
came from an earlier build and flattered live), found the **dead live volume feed and the wrong 09:15 bar** (which I
then verified), showed the core CAMARILLA / first-candle breakouts lose in *every* period across ~500 stocks, and showed
that a limit at the level suffers adverse selection. It overturned three of my early claims: ADX-FILTER's hindsight vote
nets out (small), target-before-stop is worth ~0, and "don't buy stocks more than 1% above the open" does not generalise.

**Round 2 — an audit of my code.** It found a **look-ahead bug in my replay**: I used the scan bar's midpoint
(open + close)/2 as the decision price, but that bar's close only exists after the decision, so every filter selecting
on it quietly preferred bars that closed strong. It **withdrew its own "inverted drift gate" recommendation** because of
it (the gain fell from +0.24% to about +0.04% measured causally). It also found that first-pass silently dropped 25% of
days, that limit orders could fill before they existed, and that my check periods overlapped. All were fixed before the
results above; the fixed gate chain reproduces live's gates exactly.

**Round 3 — the conclusions.** It agrees with all three, with the framing adopted here: report the two big causes
together; "none of 26 specs", with the cost arithmetic; the broken feed "may have added" losses. Nothing it could find
would flip the verdict. Its one research direction: **change the horizon** (multi-day swings), because there is no
causal way to use whole-day agreement — that information arrives after the move.

## 11 · Recommendation


**1. Don't fund the LONG system with the ₹2L.** The expected result with today's rules is about −0.24% per trade ≈
−₹480 per trade at ₹2L — roughly −₹10k a month at one trade a day; the best variant still loses ≈ −₹260 per trade. Keep
paper trading paused (it is), and put the execution-layer build on hold until there is a strategy worth executing — it
would only make these losses real.

**2. Fix the foundations** (needed for any strategy; cheap):
* Make the **backtest causal**: rank and filter on votes cast so far, enter at the next price after confirmation, stop
  before target on ambiguous bars. `analysis/replay.py` already *is* this backtester — reuse it.
* Fix the **2026 parquet time-zone bug** (the backtester silently drops 2026) and **de-duplicate** the trade history.
* **Live data parity**: repair the volume feed; build the 09:15 bar from the exchange's open/high/low, not the first tick.
* Small things the review found: NR7 never fires; slippage is charged twice in live P&L; the score thresholds in the
  time gate are dead with 3.0 weights.

**3. Research a different LONG, with a hard gate before any money.** Intraday continuation after these signals isn't
there. The most promising direction is a **multi-day swing LONG**: decide on information from the previous close (the
DAILY-BIAS trend, the chart patterns the pre-filter already detects, relative strength), enter at the next open, hold
3–10 days. Costs of 0.16% are small against 2–5% multi-day moves, and level fills, vote timing and bar parity stop
mattering. The pre-filter is already causal and history-only — a ready-made starting harness.
**Adopt a rule only if** it has the same sign in 2025, 2026 H1 and the live window with pooled t > 2, then ≥100 paper
trades on fixed data with the lower 80% confidence bound above zero — and *then* start with a fraction of the ₹2L.

*To re-run:* `python -m analysis.run_replay` (a few hours) → this notebook (a few minutes). Code in `analysis/`.